# Axis 1 — Location, intercepts and covariate centering (`31_location_and_centering`)

**Question.** Who carries the overall level of a node, and what does fitting a covariate as given (uncentered) or centered do to its coefficient?

**Design.** Covariates are always *simulated on their raw scale* (mean `XBAR` = 3), as real covariates arrive. Each data set is then fitted with the covariate as given and with it centered. What varies across data sets is the true mean of the item locations, which sets the node's level.

**Premises.**
1. Adding a constant $c$ to every $\theta$ of a dimension and to every $b$ of its nodes leaves every linear predictor $a(\theta - b + \dots)$ unchanged: one constraint is needed to fix the origin.
2. irtree imposes two: $\theta_p \sim N(0, \sigma^2)$ and $b_i \sim N(\mu_b = 0, \tau^2)$, both means fixed. A fixed *population* mean does not fix the *sample* mean: the $\theta$'s and the $b$'s can each shift together, at a prior cost of about $P c^2 / (2\sigma^2)$ and $I c^2 / (2\tau^2)$.
3. So a node has no free intercept. The likelihood pins the node's level; whatever the fitted terms do not supply (the **gap**) is carried by the free directions in proportion to their costs (the carrier calculator below).
4. An uncentered covariate supplies the constant $\bar{x}\beta$, tied to its slope. With data simulated as $\eta = \theta - b + \beta x$, an uncentered fit supplies $\bar{x}\beta$ itself, so its gap is the items' own level, $-\bar{b}$. A centered fit supplies nothing, so its gap is the whole level, $-\bar{b} + \bar{x}\beta$.

**Predictions.**
* Uncentered fits are unbiased when the item locations' true mean equals their prior mean, 0; otherwise the items' level leaks into the coefficient, mostly into an item coefficient (pinned by few items), little into a person coefficient (pinned by many persons).
* Centered fits leave the coefficients unbiased, but the item locations carry the whole level and their variance inflates by about its square.
* An explicit intercept gives the level its own parameter: then the coding no longer matters.
* The calculator predicts the coefficient biases, including how they change with more items, the trait-variance treatment, and the location-variance prior.

Every fit is a one-node probit tree (Rasch unless stated). Each setting is repeated over several simulated data sets; read the tables, not a single run.

## Setup

In [ ]:
# Install the package from this repository (editable), once per environment.
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
!{sys.executable} -m pip install -q -e "{ROOT}"

In [ ]:
import time, contextlib, io, warnings
import numpy as np, pandas as pd
from scipy.stats import norm
import irtree
warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

RUNTIME = []
@contextlib.contextmanager
def timed(step):
    t0 = time.perf_counter(); yield; dt = time.perf_counter() - t0
    RUNTIME.append({"step": step, "seconds": round(dt, 1)})
    print(f"[{step}] " + (f"{dt / 60:.1f} min" if dt >= 60 else f"{dt:.1f} s"))

## Study size
`P` persons, `I` items; `SEEDS` data sets per setting; one chain per fit (fixed warm-up). Parts 4 and 5 fit 200-item data and take most of the time (about 10 minutes each with 3 seeds); use fewer seeds for a first pass.

In [ ]:
P, I      = 1000, 20
N_ITER    = 3000
N_WARMUP  = 1000
SEEDS     = [11, 12, 13]
XBAR, XSD = 3.0, 1.0      # raw covariate mean and SD
COEF      = 0.5           # true beta and kappa
B_MEAN    = -1.0          # default true mean of the item locations (items easy)
VAGUE     = irtree.NormalPrior(mean=0.0, variance=10.0)      # coefficient prior in every fit
SPEC      = irtree.IRTreeSpec(mapping=np.array([[0], [1]]))  # one binary node
print(f"P={P} I={I} N_ITER={N_ITER} seeds={SEEDS}")

## Helpers
`simulate` draws one data set with an optional raw covariate (`cov` = "person", "item" or "both") entering as $\beta x$ or $\kappa q$. `fit` fits the one-node tree and returns the coefficients, the means of the posterior-mean item locations and traits, and the variances.

In [ ]:
def simulate(seed, cov=None, b_mean=B_MEAN, n_items=None):
    n_items = n_items or I
    rng = np.random.default_rng(seed)
    theta = rng.normal(0, 1, P)
    b = rng.normal(b_mean, 1, n_items)
    x = rng.normal(XBAR, XSD, P)
    q = rng.normal(XBAR, XSD, n_items)
    eta = theta[:, None] - b[None, :]
    if cov in ("person", "both"):
        eta = eta + COEF * x[:, None]
    if cov in ("item", "both"):
        eta = eta + COEF * q[None, :]
    Y = (rng.random((P, n_items)) < norm.cdf(eta)).astype(int)
    return dict(Y=Y, x=x, q=q, b=b, theta=theta)

def fit(Y, X=None, X_names=None, Q=None, Q_names=None, model="rasch", b_prior=None):
    cls = irtree.ProbitRaschNode if model == "rasch" else irtree.ProbitTwoPLNode
    node = cls(person_covariates=True if X is not None else None,
               item_covariates=True if Q is not None else None)
    data = irtree.expand_data(Y, SPEC, categories_start=0, X=X, X_names=X_names, Q=Q, Q_names=Q_names)
    priors = irtree.build_default_priors(SPEC, [node])
    priors.set("beta", VAGUE); priors.set("kappa", VAGUE)
    if b_prior is not None:
        priors.set("b", b_prior)
    with contextlib.redirect_stdout(io.StringIO()):
        f = irtree.irtree_fit(data, SPEC, [node], priors=priors, n_iter=N_ITER, n_warmup=N_WARMUP,
                              warmup_mode="fixed", n_chains=1, seed=1, progress=False)
    s = f.summary_df()
    out = {n: s.loc[n, "mean"] for n in s.index if n.startswith(("beta_", "kappa_"))}
    out |= {f"{n} sd": s.loc[n, "sd"] for n in s.index if n.startswith(("beta_", "kappa_"))}
    out["mean b"] = s.loc[[n for n in s.index if n.startswith("b_1_")], "mean"].mean()
    out["mean theta"] = s.loc[[n for n in s.index if n.startswith("theta_")], "mean"].mean()
    out["b.variance"] = s.loc["b.variance_1", "mean"] if "b.variance_1" in s.index else np.nan
    out["sigma2"] = s.loc["sigma2_1", "mean"]
    return out

def col(v):
    return v[:, None]

## The carrier calculator: one formula for every case
**Why a formula.** Covariates, numbers of persons and items, estimated or fixed variances (and their values), Rasch or 2PL, centering, intercept: a full cross table is large, but every factor acts through the same mechanism. Under the probit link and given the latent responses, the node is a linear model with crossed random effects, so the location direction can be worked out.

**Premises.**
1. The data pin the node's level (its grand mean on the latent scale) with high precision ($P \times I$ cells).
2. The slope of each covariate is pinned by its *deviations*: $\beta$ by the person means, with precision $\pi_\beta = P\,s_x^2 / (\sigma^2 + 1/I)$; $\kappa$ by the item means, with precision $\pi_\kappa = I\,s_q^2 / (\tau^2 + 1/P)$.
3. The gap $\Delta$ (the part of the level the fitted terms do not supply at the slope-implied coefficients) is carried by the free directions: the trait mean, the location mean, each uncentered coefficient (through its covariate's mean), and an intercept if there is one.

**Result.** Each carrier $j$ has a *cost* $c_j$, the prior variance of the direction it moves along, and takes the share $w_j = c_j / \sum_k c_k$ of the gap:

| Carrier | Cost $c_j$ | Offset it takes |
|---|---|---|
| trait mean $\bar\theta$ | $\sigma^2 / P$ | $\bar\theta \approx w_\theta \Delta$ |
| location mean $\bar b$ | $\tau^2 / I$ | $\bar b \approx -w_b \Delta$ |
| person coefficient | $\bar{x}^2 / \pi_\beta$ | $\beta - \hat\beta \approx w_\beta \Delta / \bar x$ |
| item coefficient | $\bar{q}^2 / \pi_\kappa$ | $\kappa - \hat\kappa \approx w_\kappa \Delta / \bar q$ |
| intercept | its prior variance (10: almost free) | most of $\Delta$ |

**What the cross-table factors become.** Every factor enters only through $\Delta$ (the coding and the true level) or a cost.
* With the traits' cost negligible, the item coefficient's share is $\bar q^2 / (s_q^2 + \bar q^2)$, so its bias is $\Delta\,\bar q / (s_q^2 + \bar q^2)$: **$I$ and $\tau^2$ cancel**. More items or a different location prior do not change it; only the covariate's mean-to-SD ratio does.
* The person coefficient's share grows roughly with $I/P$: more items make the item locations an expensive carrier, so the person coefficient takes more.
* An estimated variance enters at its fitted value, which itself inflates when its carrier takes an offset; a fixed one keeps its value.
* Rasch versus 2PL changes only $\sigma^2$ in the formula (estimated versus 1); the slopes themselves are not in it.

**Limits.** Exact for a probit node with the latent responses given and every cell relevant; approximate otherwise (binary responses carry less information than latent ones, logit nodes, trees where nodes see only some cells, 2PL slopes).

In [ ]:
def carriers(P_, I_, sigma2, tau2, xbar=0.0, xsd=1.0, qbar=0.0, qsd=1.0, gap=0.0, intercept_var=None):
    # Shares of the gap, and predicted offsets and coefficient biases (table above).
    cost = {"theta": sigma2 / P_, "b": tau2 / I_}
    if xbar:
        cost["beta"] = xbar**2 / (P_ * xsd**2 / (sigma2 + 1 / I_))
    if qbar:
        cost["kappa"] = qbar**2 / (I_ * qsd**2 / (tau2 + 1 / P_))
    if intercept_var:
        cost["intercept"] = intercept_var
    S = sum(cost.values())
    share = {k: v / S for k, v in cost.items()}
    pred = {"mean theta": share["theta"] * gap, "mean b": -share["b"] * gap}
    if "beta" in share:
        pred["beta bias"] = share["beta"] * gap / xbar
    if "kappa" in share:
        pred["kappa bias"] = share["kappa"] * gap / qbar
    return share, pred

# Example: item covariate fitted as given, items with mean location -1 (gap = +1)
share, pred = carriers(P, I, sigma2=1.0, tau2=1.2, qbar=XBAR, gap=1.0)
print({k: round(v, 3) for k, v in share.items()})
print({k: round(v, 3) for k, v in pred.items()})

## Part 1 — Who carries the level when nothing else can
No covariates; items with mean location `B_MEAN`, so the level is about $-$`B_MEAN`. Prediction: the traits take the share $(\sigma^2/P)/(\sigma^2/P + \tau^2/I)$, the items the rest, and the location variance inflates.

In [ ]:
share_theta = (1 / P) / (1 / P + 1 / I)
print(f"predicted share of the level carried by the traits: {share_theta:.1%}")
rows = []
with timed("Part 1"):
    for s in SEEDS:
        d = simulate(s)
        L = d["theta"].mean() - d["b"].mean()
        r = fit(d["Y"])
        rows.append({"seed": s, "true level": L, "true mean b": d["b"].mean(), "fit mean b": r["mean b"],
                     "fit mean theta": r["mean theta"], "share carried by traits": r["mean theta"] / L,
                     "true var b": d["b"].var(ddof=1), "fit b.variance": r["b.variance"]})
part1 = pd.DataFrame(rows).round(3)
part1

**Read.** The fitted mean of the $b$'s is close to the true one (the $b$'s carry the level), the traits carry about the predicted share, and the location variance is inflated above the true spread because the prior $N(0, \tau^2)$ must also cover the offset of the $b$'s from 0.

## Part 2 — Raw covariate, fitted as given or centered, at several item levels
Data: $\eta = \theta - b + 0.5\,x$ (person covariate) or $\theta - b + 0.5\,q$ (item covariate), with the item locations' true mean in `B_MEANS`. Gap: $-\bar b$ when fitted as given, $-\bar b + 0.5\,\bar{x}$ when centered. The `pred coef` column is the calculator's prediction for the fit as given; centered fits have no coefficient carrier, so their coefficient should stay near 0.5 while the locations absorb the gap.

In [ ]:
B_MEANS = [-1.0, 0.0, 0.5, 1.5]
rows = []
with timed("Part 2"):
    for bm in B_MEANS:
        for s in SEEDS:
            for cov in ("person", "item"):
                d = simulate(s, cov=cov, b_mean=bm)
                v = d["x"] if cov == "person" else d["q"]
                for coding in ("as given", "centered"):
                    vv = v if coding == "as given" else v - v.mean()
                    if cov == "person":
                        r = fit(d["Y"], X=col(vv), X_names=["x"]); c = "beta_1_x"
                    else:
                        r = fit(d["Y"], Q=col(vv), Q_names=["q"]); c = "kappa_1_q"
                    gap = -d["b"].mean() + d["theta"].mean() + (0.0 if coding == "as given" else COEF * v.mean())
                    pred = np.nan
                    if coding == "as given":
                        kw = dict(xbar=v.mean(), xsd=v.std()) if cov == "person" else dict(qbar=v.mean(), qsd=v.std())
                        _, pr = carriers(P, I, r["sigma2"], r["b.variance"], gap=gap, **kw)
                        pred = COEF + pr["beta bias" if cov == "person" else "kappa bias"]
                    rows.append({"true mean b": bm, "seed": s, "covariate": cov, "coding": coding, "gap": gap,
                                 "coef": r[c], "pred coef": pred, "coef sd": r[c + " sd"],
                                 "fit mean b": r["mean b"], "fit b.variance": r["b.variance"],
                                 "true var b": d["b"].var(ddof=1)})
part2 = pd.DataFrame(rows).round(3)
part2.groupby(["covariate", "coding", "true mean b"])[
    ["gap", "coef", "pred coef", "coef sd", "fit b.variance", "true var b"]].mean().round(3)

**Read.**
* As given: the coefficient is unbiased when the true mean of $b$ is 0 (gap 0). Otherwise the item coefficient's bias follows the gap, sign included, and the calculator's `pred coef` tracks it; the person coefficient moves little.
* Centered: the person coefficient stays near 0.5 at every level, but the location variance inflates by about the square of the gap. The item coefficient has no systematic bias but a very wide SD (0.3 to 0.7 in the reference run), so its average over three data sets is itself noisy; the SD shrinks as the gap goes to 0. The inflation vanishes only where the gap is 0 (true mean of $b$ = 1.5, where $-\bar b + 0.5 \times 3 = 0$).
* Reference run (3 data sets): item coefficient as given 0.78, 0.48, 0.33, 0.02 at true mean $b$ = −1, 0, 0.5, 1.5, against predictions 0.78, 0.48, 0.33, 0.03.

In [ ]:
part2   # every data set

## Part 3 — An explicit intercept
Data with both raw covariates, items with mean location `B_MEAN`. Fit with the person covariate and, among the item covariates, a column of ones (`kappa_1_intercept`, prior N(0, 10)), as given and centered. Prediction: the intercept carries the gap; the locations stay centered, their variance near the truth, and $\beta$, $\kappa$ are the same in both codings.

In [ ]:
rows = []
with timed("Part 3"):
    for s in SEEDS:
        d = simulate(s, cov="both")
        ones = np.ones(I)
        for coding in ("as given", "centered"):
            x = d["x"] if coding == "as given" else d["x"] - d["x"].mean()
            q = d["q"] if coding == "as given" else d["q"] - d["q"].mean()
            r = fit(d["Y"], X=col(x), X_names=["x"], Q=np.column_stack([ones, q]), Q_names=["intercept", "q"])
            rows.append({"seed": s, "coding": coding, "beta": r["beta_1_x"], "kappa": r["kappa_1_q"],
                         "kappa sd": r["kappa_1_q sd"], "intercept": r["kappa_1_intercept"],
                         "fit mean b": r["mean b"], "fit b.variance": r["b.variance"],
                         "true var b": d["b"].var(ddof=1)})
part3 = pd.DataFrame(rows).round(3)
display(part3)
part3.groupby("coding")[["beta", "kappa", "kappa sd", "intercept", "fit mean b", "fit b.variance", "true var b"]].mean().round(3)

**Read.** Same $\beta$ and $\kappa$ in both codings, locations centered near 0 with the right variance. The intercept is about $-\bar b + 0.5\bar x + 0.5\bar q$ when centered (the whole level) and about $-\bar b$ as given. $\kappa$ still varies across data sets with a wide SD: with 20 items the sampled $q$ can correlate with the locations by chance. That is honest sampling variability.

## Part 4 — Many more items, persons fixed
Raw covariate fitted as given, items with mean location `B_MEAN` (gap about +1), with 20 and 200 items. Calculator predictions: the item coefficient's bias does not shrink (I cancels); the person coefficient's bias grows; the traits take a larger share of the gap.

In [ ]:
rows = []
with timed("Part 4"):
    for n_items in (20, 200):
        for s in SEEDS:
            for cov in ("person", "item"):
                d = simulate(s, cov=cov, n_items=n_items)
                gap = d["theta"].mean() - d["b"].mean()
                if cov == "person":
                    r = fit(d["Y"], X=col(d["x"]), X_names=["x"]); c, kw = "beta_1_x", dict(xbar=d["x"].mean(), xsd=d["x"].std())
                else:
                    r = fit(d["Y"], Q=col(d["q"]), Q_names=["q"]); c, kw = "kappa_1_q", dict(qbar=d["q"].mean(), qsd=d["q"].std())
                share, pr = carriers(P, n_items, r["sigma2"], r["b.variance"], gap=gap, **kw)
                rows.append({"I": n_items, "seed": s, "covariate": cov, "coef": r[c],
                             "pred coef": COEF + pr["beta bias" if cov == "person" else "kappa bias"],
                             "fit mean theta": r["mean theta"], "pred mean theta": d["theta"].mean() + pr["mean theta"],
                             "fit b.variance": r["b.variance"]})
part4 = pd.DataFrame(rows).round(3)
display(part4)
part4.groupby(["covariate", "I"])[["coef", "pred coef", "fit mean theta", "pred mean theta"]].mean().round(3)

**Read.** Compare `coef` with `pred coef`, and the averages across `I`. Reference run: item coefficient 0.78 at 20 items and 0.80 at 200 (predicted 0.78, 0.81); person coefficient 0.56 and 0.73 (predicted 0.53, 0.70). The item bias does not shrink with more items, the person bias grows; the calculator is close for the item coefficient and slightly underestimates the person bias.

## Part 5 — Trait variance estimated (Rasch) or fixed at 1 (2PL)
Rasch data (every slope 1) with the raw person covariate, fitted as given as a Rasch node (trait variance estimated) and as a 2PL node (trait variance fixed at 1, slopes free); 20 and 200 items. Both models contain the truth. The calculator has no slopes: in it, Rasch and 2PL differ only through $\sigma^2$, which is close to 1 either way, so it predicts no difference. A difference here would come from the slopes.

In [ ]:
rows = []
with timed("Part 5"):
    for n_items in (20, 200):
        for s in SEEDS:
            d = simulate(s, cov="person", n_items=n_items)
            for model in ("rasch", "2pl"):
                r = fit(d["Y"], X=col(d["x"]), X_names=["x"], model=model)
                rows.append({"I": n_items, "seed": s, "model": model, "beta": r["beta_1_x"],
                             "beta sd": r["beta_1_x sd"], "sigma2": r["sigma2"], "fit mean theta": r["mean theta"],
                             "fit mean b": r["mean b"], "fit b.variance": r["b.variance"]})
part5 = pd.DataFrame(rows).round(3)
display(part5)
part5.groupby(["I", "model"])[["beta", "beta sd", "sigma2", "fit mean theta", "fit mean b", "fit b.variance"]].mean().round(3)

**Read.** In the reference run the 2PL estimate of $\beta$ was about 0.03 to 0.04 below the Rasch estimate at both 20 and 200 items, and an earlier run with a gap of the opposite sign showed a shift in the same direction. A shift that does not follow the sign of the gap is not a carrier effect. It is more likely a scale effect: $\beta$ is measured in trait units, and with free slopes that unit is set by the fixed trait variance together with the slope prior. Treat this as an open question for the identification paper.

## Part 6 — Location variance fixed low, fixed high, or estimated
Raw item covariate fitted as given, 20 items, items with mean location `B_MEAN`. The location prior is $N(0, \tau^2)$ with $\tau^2$ fixed at 0.1, fixed at 10, or estimated (the default). Calculator prediction: $\tau^2$ cancels from the item coefficient's share, so its bias is the same in all three; only its SD changes (it is pinned through the location prior, so a smaller $\tau^2$ gives a smaller SD).

In [ ]:
rows = []
with timed("Part 6"):
    for s in SEEDS:
        d = simulate(s, cov="item")
        gap = d["theta"].mean() - d["b"].mean()
        for lab, prior in (("fixed 0.1", irtree.NormalPrior(mean=0.0, variance=0.1)),
                           ("fixed 10", irtree.NormalPrior(mean=0.0, variance=10.0)),
                           ("estimated", None)):
            r = fit(d["Y"], Q=col(d["q"]), Q_names=["q"], b_prior=prior)
            tau2 = {"fixed 0.1": 0.1, "fixed 10": 10.0}.get(lab, r["b.variance"])
            share, pr = carriers(P, I, r["sigma2"], tau2, qbar=d["q"].mean(), qsd=d["q"].std(), gap=gap)
            rows.append({"seed": s, "tau2": lab, "kappa": r["kappa_1_q"], "pred kappa": COEF + pr["kappa bias"],
                         "kappa sd": r["kappa_1_q sd"], "share kappa": share["kappa"],
                         "fit mean b": r["mean b"], "true mean b": d["b"].mean()})
part6 = pd.DataFrame(rows).round(3)
display(part6)
part6.groupby("tau2")[["kappa", "pred kappa", "kappa sd", "share kappa", "fit mean b"]].mean().round(3)

**Read.** Similar `kappa` in all three rows, tracked by `pred kappa`, with very different SDs: with $\tau^2$ = 0.1 the item coefficient looks precise while being just as biased. A narrow posterior is not evidence of no bias. (With $\tau^2$ = 0.1 the fitted value was somewhat below the prediction in the reference run, 0.72 against 0.77: the approximation is weakest when the locations are held tight.)

## Conclusion
* Fixing both the trait mean and the location prior mean imposes one constraint more than the location indeterminacy needs, so a node has no free intercept. The gap between the node's level and what the fitted terms supply goes to the free directions in proportion to their costs.
* Fitted as given, a raw covariate is unbiased only when the item locations' true mean equals their prior mean. Otherwise an item coefficient absorbs most of the gap, with a bias $\Delta\,\bar q/(s_q^2 + \bar q^2)$ that more items or a different location prior do not remove; a person coefficient absorbs little, more so as items outnumber persons.
* Centered, the coefficients are unbiased but the location variance inflates by about the square of the level.
* An explicit intercept (or an estimated $\mu_b$) gives the level its own parameter; with it, the coding does not matter.

## Runtime

In [ ]:
rt = pd.DataFrame(RUNTIME)
rt.loc[len(rt)] = {"step": "TOTAL", "seconds": rt["seconds"].sum()}
rt